In [1]:
import scipy.io as sio
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import urllib.request
import os

print("Loading SVHN dataset...")

# Create directory
os.makedirs('/tmp/svhn', exist_ok=True)

# Download if not exists
train_file = '/tmp/svhn/train_32x32.mat'
test_file = '/tmp/svhn/test_32x32.mat'

if not os.path.exists(train_file):
    print("Downloading train set...")
    urllib.request.urlretrieve(
        'http://ufldl.stanford.edu/housenumbers/train_32x32.mat',
        train_file
    )
    print("✓ Train downloaded")

if not os.path.exists(test_file):
    print("Downloading test set...")
    urllib.request.urlretrieve(
        'http://ufldl.stanford.edu/housenumbers/test_32x32.mat',
        test_file
    )
    print("✓ Test downloaded")

# Load .mat files
print("Loading data from .mat files...")
train_data = sio.loadmat(train_file)
test_data = sio.loadmat(test_file)

# Extract arrays
X_train = train_data['X']
y_train = train_data['y']
X_test = test_data['X']
y_test = test_data['y']

# Transpose: (H, W, C, N) -> (N, H, W, C)
X_train = np.transpose(X_train, (3, 0, 1, 2))
X_test = np.transpose(X_test, (3, 0, 1, 2))

# Fix labels: 10 -> 0
y_train[y_train == 10] = 0
y_test[y_test == 10] = 0

print(f"✓ Train: {X_train.shape}")
print(f"✓ Test: {X_test.shape}")

# Normalize
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Categorical
y_cat_train = to_categorical(y_train, 10)
y_cat_test = to_categorical(y_test, 10)

print("✓ SVHN loaded successfully!")

2026-03-23 16:10:56.938228: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774282257.131842      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774282257.193707      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774282257.685513      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774282257.685563      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774282257.685566      23 computation_placer.cc:177] computation placer alr

Loading SVHN dataset...
✓ Train downloaded
✓ Test downloaded
Loading data from .mat files...
✓ Train: (73257, 32, 32, 3)
✓ Test: (26032, 32, 32, 3)
✓ SVHN loaded successfully!


In [2]:
# ============================================================

!pip uninstall codecarbon -y
!pip install codecarbon==2.3.4 --quiet
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from codecarbon import EmissionsTracker
import psutil
import subprocess
import time
import os
import gc

# GPU config
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Configuration
SEED = 42
BATCH_SIZE = 32
EPOCHS = 50
NUM_RUNS = 10
VARIANT = "baseline_svhn"
CSV_FILE = "baseline_svhn_results.csv"

print("="*70)
print("M3 - CNN BASELINE ON SVHN")
print("="*70)

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("⚠ No GPU")
print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.6/181.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.9 MB/s eta 0:00:00
M3 - CNN BASELINE ON SVHN
✓ GPU: 1 device(s)



In [3]:
# ============================================================

def build_model():
    """Build 3-block CNN"""
    model = Sequential([
        # Block 1
        Conv2D(32, (3, 3), input_shape=(32, 32, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.25),

        # Block 2
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.25),

        # Block 3
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPool2D((2, 2)),
        Dropout(0.25),

        # Dense
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.25),
        Dense(10, activation='softmax')
    ])
    return model

print("✓ Model ready")

✓ Model ready


In [4]:
# ============================================================

class MemoryTracker:
    """Track memory using nvidia-smi"""
    def __init__(self, phase_name):
        self.phase_name = phase_name
        self.process = psutil.Process()
        self.start_cpu = 0
        self.start_gpu = 0
        self.peak_cpu = 0
        self.peak_gpu = 0

    def get_gpu_memory(self):
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,nounits,noheader'],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                timeout=5
            )
            if result.returncode == 0:
                memory_str = result.stdout.decode('utf-8').strip().split('\n')[0]
                return float(memory_str)
            return 0
        except:
            return 0

    def start(self):
        self.start_cpu = self.process.memory_info().rss / (1024**2)
        self.start_gpu = self.get_gpu_memory()

    def stop(self):
        end_cpu = self.process.memory_info().rss / (1024**2)
        end_gpu = self.get_gpu_memory()
        self.peak_cpu = end_cpu - self.start_cpu
        self.peak_gpu = end_gpu - self.start_gpu
        return self.peak_cpu, self.peak_gpu

print("✓ MemoryTracker ready")

✓ MemoryTracker ready


In [5]:
# ============================================================

all_results = []

for run_id in range(1, NUM_RUNS + 1):
    print(f"\n{'='*70}")
    print(f"RUN {run_id}/{NUM_RUNS}")
    print(f"{'='*70}\n")

    try:
        # ❌ SMELL INJECTION: Improper Model Reuse (M3 Logic)
        # REMOVED: tf.keras.backend.clear_session()
        # This causes the "Accumulation of Computation Graphs" 
        
        gc.collect()

        np.random.seed(SEED + run_id)
        tf.random.set_seed(SEED + run_id)

        # ❌ SMELL INJECTION: Wasteful Model Creation (as seen in Milestone 2 style)
        # --- Improper Model Reuse (SMELL INJECTION) ---
        print("Building model...")
        temp_model = build_model() # ❌ waste
        del temp_model
        gc.collect()

        model = build_model()     # ✅ actual model
        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy',
                    tf.keras.metrics.Precision(name='precision'),
                    tf.keras.metrics.Recall(name='recall')]
        )

        # Early Stopping
        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
            verbose=1
        )

        # CRITICAL: Create TF Dataset for GPU efficiency
        train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_cat_train))
        train_ds = train_ds.shuffle(10000, seed=SEED + run_id).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_cat_test))
        test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

        # Training
        print(f"\nTraining ({EPOCHS} epochs)...")

        mem_tracker_train = MemoryTracker("train")
        mem_tracker_train.start()

        tracker_train = EmissionsTracker(
            project_name=f"{VARIANT}_train_{run_id}",
            output_dir=".",
            output_file="codecarbon_train.csv",
            log_level="error"
        )
        tracker_train.start()

        train_start = time.time()

        history = model.fit(
            train_ds,  # TF Dataset!
            epochs=EPOCHS,
            validation_data=test_ds,
            callbacks=[early_stop],
            verbose=2
        )

        train_time = time.time() - train_start
        train_emissions = tracker_train.stop()
        train_energy = getattr(tracker_train.final_emissions_data, "energy_consumed", None)

        cpu_train, gpu_train = mem_tracker_train.stop()

        print(f"✓ Training: {train_time:.1f}s")
        print(f"  CPU: +{cpu_train:.1f} MB | GPU: +{gpu_train:.1f} MB")

        # Evaluation
        print("\nEvaluating...")

        mem_tracker_eval = MemoryTracker("eval")
        mem_tracker_eval.start()

        tracker_eval = EmissionsTracker(
            project_name=f"{VARIANT}_eval_{run_id}",
            output_dir=".",
            output_file="codecarbon_eval.csv",
            log_level="error"
        )
        tracker_eval.start()

        eval_start = time.time()
        results_eval = model.evaluate(test_ds, verbose=0)
        eval_time = time.time() - eval_start

        eval_emissions = tracker_eval.stop()
        eval_energy = getattr(tracker_eval.final_emissions_data, "energy_consumed", None)

        cpu_eval, gpu_eval = mem_tracker_eval.stop()

        test_acc = results_eval[1]
        test_prec = results_eval[2]
        test_rec = results_eval[3]

        print(f"✓ Acc: {test_acc:.4f} | Prec: {test_prec:.4f} | Rec: {test_rec:.4f}")

        # Save
        results = {
            'variant': VARIANT,
            'run_id': run_id,
            'train_time_sec': round(train_time, 4),
            'eval_time_sec': round(eval_time, 4),
            'cpu_peak_train_mb': round(cpu_train, 2),
            'gpu_peak_train_mb': round(gpu_train, 2) if gpu_train > 0 else "NA",
            'cpu_peak_eval_mb': round(cpu_eval, 2),
            'gpu_peak_eval_mb': round(gpu_eval, 2) if gpu_eval > 0 else "NA",
            'train_energy_kwh': round(train_energy, 6) if train_energy else "NA",
            'train_co2_kg': round(train_emissions, 6) if train_emissions else "NA",
            'eval_energy_kwh': round(eval_energy, 6) if eval_energy else "NA",
            'eval_co2_kg': round(eval_emissions, 6) if eval_emissions else "NA",
            'test_accuracy': round(test_acc, 4),
            'test_precision': round(test_prec, 4),
            'test_recall': round(test_rec, 4),
        }

        all_results.append(results)
        df = pd.DataFrame(all_results)
        df.to_csv(CSV_FILE, index=False)

        print(f"✓ Saved to {CSV_FILE}\n")

        del model
        gc.collect()

    except Exception as e:
        print(f"✗ ERROR: {e}")
        import traceback
        traceback.print_exc()

print("\n" + "="*70)
print("COMPLETE")
print("="*70)
if all_results:
    df = pd.DataFrame(all_results)
    print(f"✓ {len(all_results)}/{NUM_RUNS} runs")


RUN 1/10

Building model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1774282308.931712      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5



Training (50 epochs)...
Epoch 1/50


I0000 00:00:1774282324.919065      79 service.cc:152] XLA service 0x7ada44002c40 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774282324.919107      79 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1774282325.806856      79 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1774282332.852212      79 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2290/2290 - 36s - 16ms/step - accuracy: 0.6767 - loss: 0.9737 - precision: 0.8703 - recall: 0.5840 - val_accuracy: 0.8631 - val_loss: 0.4472 - val_precision: 0.9172 - val_recall: 0.8175
Epoch 2/50
2290/2290 - 15s - 6ms/step - accuracy: 0.8757 - loss: 0.4261 - precision: 0.9238 - recall: 0.8385 - val_accuracy: 0.8969 - val_loss: 0.3587 - val_precision: 0.9364 - val_recall: 0.8622
Epoch 3/50
2290/2290 - 15s - 6ms/step - accuracy: 0.9007 - loss: 0.3464 - precision: 0.9378 - recall: 0.8710 - val_accuracy: 0.9190 - val_loss: 0.2874 - val_precision: 0.9446 - val_recall: 0.8994
Epoch 4/50
2290/2290 - 15s - 6ms/step - accuracy: 0.9155 - loss: 0.2959 - precision: 0.9462 - recall: 0.8917 - val_accuracy: 0.9345 - val_loss: 0.2479 - val_precision: 0.9573 - val_recall: 0.9125
Epoch 5/50
2290/2290 - 15s - 7ms/step - accuracy: 0.9238 - loss: 0.2671 - precision: 0.9508 - recall: 0.9026 - val_accuracy: 0.9347 - val_loss: 0.2442 - val_precision: 0.9554 - val_recall: 0.9178
Epoch 6/50
2290/2290 - 15s - 7

...


Training (50 epochs)...
Epoch 1/50
2290/2290 - 34s - 15ms/step - accuracy: 0.6907 - loss: 0.9434 - precision: 0.8730 - recall: 0.6017 - val_accuracy: 0.8781 - val_loss: 0.4178 - val_precision: 0.9341 - val_recall: 0.8291
Epoch 2/50
2290/2290 - 15s - 7ms/step - accuracy: 0.8806 - loss: 0.4148 - precision: 0.9263 - recall: 0.8441 - val_accuracy: 0.9104 - val_loss: 0.3132 - val_precision: 0.9447 - val_recall: 0.8819
Epoch 3/50
2290/2290 - 15s - 7ms/step - accuracy: 0.9017 - loss: 0.3435 - precision: 0.9384 - recall: 0.8715 - val_accuracy: 0.8695 - val_loss: 0.4392 - val_precision: 0.9241 - val_recall: 0.8274
Epoch 4/50
2290/2290 - 15s - 7ms/step - accuracy: 0.9156 - loss: 0.3002 - precision: 0.9462 - recall: 0.8901 - val_accuracy: 0.9173 - val_loss: 0.2949 - val_precision: 0.9472 - val_recall: 0.8894
Epoch 5/50
2290/2290 - 15s - 6ms/step - accuracy: 0.9230 - loss: 0.2715 - precision: 0.9511 - recall: 0.9010 - val_accuracy: 0.9378 - val_loss: 0.2379 - val_precision: 0.9547 - val_recall: 0

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training (50 epochs)...
Epoch 1/50
2290/2290 - 34s - 15ms/step - accuracy: 0.6833 - loss: 0.9592 - precision: 0.8737 - recall: 0.5900 - val_accuracy: 0.8905 - val_loss: 0.3878 - val_precision: 0.9305 - val_recall: 0.8590
Epoch 2/50
2290/2290 - 15s - 7ms/step - accuracy: 0.8781 - loss: 0.4215 - precision: 0.9247 - recall: 0.8398 - val_accuracy: 0.9092 - val_loss: 0.3193 - val_precision: 0.9427 - val_recall: 0.8828
Epoch 3/50
2290/2290 - 15s - 7ms/step - accuracy: 0.9009 - loss: 0.3506 - precision: 0.9386 - recall: 0.8706 - val_accuracy: 0.9216 - val_loss: 0.2752 - val_precision: 0.9532 - val_recall: 0.8950
Epoch 4/50
2290/2290 - 15s - 6ms/step - accuracy: 0.9132 - loss: 0.3045 - precision: 0.9451 - recall: 0.8882 - val_accuracy: 0.9320 - val_loss: 0.2533 - val_precision: 0.9567 - val_recall: 0.9089
Epoch 5/50
2290/2290 - 15s - 6ms/step - accuracy: 0.9229 - loss: 0.2739 - precision: 0.9514 - recall: 0.9010 - val_accuracy: 0.9134 - val_loss: 0.3062 - val_precision: 0.9427 - val_recall: 0

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Training (50 epochs)...
Epoch 1/50
2290/2290 - 34s - 15ms/step - accuracy: 0.6484 - loss: 1.0525 - precision: 0.8675 - recall: 0.5532 - val_accuracy: 0.8720 - val_loss: 0.4319 - val_precision: 0.9226 - val_recall: 0.8303
Epoch 2/50
2290/2290 - 15s - 7ms/step - accuracy: 0.8741 - loss: 0.4327 - precision: 0.9239 - recall: 0.8354 - val_accuracy: 0.8954 - val_loss: 0.3594 - val_precision: 0.9318 - val_recall: 0.8639
Epoch 3/50
2290/2290 - 15s - 7ms/step - accuracy: 0.8983 - loss: 0.3551 - precision: 0.9362 - recall: 0.8668 - val_accuracy: 0.8763 - val_loss: 0.4179 - val_precision: 0.9317 - val_recall: 0.8344
Epoch 4/50
2290/2290 - 15s - 7ms/step - accuracy: 0.9124 - loss: 0.3074 - precision: 0.9457 - recall: 0.8877 - val_accuracy: 0.9241 - val_loss: 0.2748 - val_precision: 0.9483 - val_recall: 0.9060
Epoch 5/50
2290/2290 - 15s - 7ms/step - accuracy: 0.9232 - loss: 0.2748 - precision: 0.9516 - recall: 0.9001 - val_accuracy: 0.9086 - val_loss: 0.3135 - val_precision: 0.9415 - val_recall: 0